# Búsqueda de rutas en el Metro de la Ciudad de México con DFS y BFS

## Contexto

En esta práctica se modelará una parte de la red del Metro de la Ciudad de México como un **grafo ponderado**, donde cada estación será representada como un nodo y cada conexión entre estaciones consecutivas será una arista.

El peso de cada arista corresponderá a la **distancia en metros entre estaciones**, utilizando como referencia la información oficial publicada por el Sistema de Transporte Colectivo Metro de la Ciudad de México.

El objetivo será encontrar rutas entre diferentes estaciones utilizando dos algoritmos de búsqueda:

- **DFS (Depth First Search)** o búsqueda en profundidad.
- **BFS (Breadth First Search)** o búsqueda en anchura.

Los recorridos a analizar serán:

1. Cuatro Caminos → Pantitlán
2. Politécnico → Tasqueña
3. Zapata → Oceanía

Para la implementación se utilizará como base la estructura de clases y métodos vista en clase, particularmente las clases `Problem`, `GraphProblem` y `Node`, así como los algoritmos DFS y BFS.

Se modelarán únicamente las líneas necesarias para resolver los recorridos planteados, con el fin de simplificar el grafo sin perder las conexiones y transbordos requeridos.

## Fuentes utilizadas

- Repositorio de la materia de Sistemas Inteligentes en GitHub.
- Página oficial del Metro de la Ciudad de México para consultar las longitudes de interestación.

## Herramientas utilizadas

- Python
- Jupyter Notebook
- Visual Studio Code

# 1. Formulación del problema

In [261]:
# Clase abstracta
class Problem:
    def __init__(self, initial, goal):
        self.initial = initial  # Estado inicial
        self.goal = goal        # Meta

    def actions(self, state):
        raise NotImplementedError

    # Función de transición
    def result(self, state, action):
        raise NotImplementedError

    # Función de desempeño
    def is_goal(self, state):
        return self.goal == state

    def action_cost(self, state1, action, state2):
        return 1

    def h(self, state):
        return 0

In [262]:
class GraphProblem(Problem):
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph
    
    def actions(self, state):
        lista = []
        for key in self.graph[state].keys():
            lista.append(key)
        return lista

    # Función de transición
    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        return self.graph[state1][state2]

In [263]:
metro_cdmx = {
    # Línea 1
    "Observatorio": {
        "Tacubaya": 1262
    },

    "Tacubaya": {
        "Observatorio": 1262,
        "Juanacatlán": 1158,
        "Constituyentes": 1005,
        "San Pedro de los Pinos": 1084,
        "Patriotismo": 1133
    },

    "Juanacatlán": {
        "Tacubaya": 1158,
        "Chapultepec": 973
    },

    "Chapultepec": {
        "Juanacatlán": 973,
        "Sevilla": 501
    },

    "Sevilla": {
        "Chapultepec": 501,
        "Insurgentes": 645
    },

    "Insurgentes": {
        "Sevilla": 645,
        "Cuauhtémoc": 793
    },

    "Cuauhtémoc": {
        "Insurgentes": 793,
        "Balderas": 409
    },

    "Balderas": {
        "Cuauhtémoc": 409,
        "Salto del Agua": 458,
        "Juárez": 659,
        "Niños Héroes": 665
    },

    "Salto del Agua": {
        "Balderas": 458,
        "Isabel la Católica": 445,
        "San Juan de Letrán": 292,
        "Doctores": 564
    },

    "Isabel la Católica": {
        "Salto del Agua": 445,
        "Pino Suárez": 382
    },

    "Pino Suárez": {
        "Isabel la Católica": 382,
        "Merced": 745,
        "Zócalo/Tenochtitlan": 745,
        "San Antonio Abad": 817
    },

    "Merced": {
        "Pino Suárez": 745,
        "Candelaria": 698
    },

    "Candelaria": {
        "Merced": 698,
        "San Lázaro": 866,
        "Fray Servando": 633,
        "Morelos": 1062
    },

    "San Lázaro": {
        "Candelaria": 866,
        "Moctezuma": 478,
        "Ricardo Flores Magón": 907,
        "Morelos": 1296
    },

    "Moctezuma": {
        "San Lázaro": 478,
        "Balbuena": 703
    },

    "Balbuena": {
        "Moctezuma": 703,
        "Boulevard Puerto Aéreo": 595
    },

    "Boulevard Puerto Aéreo": {
        "Balbuena": 595,
        "Gómez Farías": 611
    },

    "Gómez Farías": {
        "Boulevard Puerto Aéreo": 611,
        "Zaragoza": 762
    },

    "Zaragoza": {
        "Gómez Farías": 762,
        "Pantitlán": 1320
    },

    "Pantitlán": {
        "Zaragoza": 1320,
        "Hangares": 1644,
        "Puebla": 1380,
        "Agrícola Oriental": 1409
    },


    # Línea 2
    "Cuatro Caminos": {
        "Panteones": 1639
    },

    "Panteones": {
        "Cuatro Caminos": 1639,
        "Tacuba": 1416
    },

    "Tacuba": {
        "Panteones": 1416,
        "Cuitláhuac": 637,
        "Refinería": 1295,
        "San Joaquín": 1433
    },

    "Cuitláhuac": {
        "Tacuba": 637,
        "Popotla": 620
    },

    "Popotla": {
        "Cuitláhuac": 620,
        "Colegio Militar": 462
    },

    "Colegio Militar": {
        "Popotla": 462,
        "Normal": 516
    },

    "Normal": {
        "Colegio Militar": 516,
        "San Cosme": 657
    },

    "San Cosme": {
        "Normal": 657,
        "Revolución": 537
    },

    "Revolución": {
        "San Cosme": 537,
        "Hidalgo": 587
    },

    "Hidalgo": {
    "Revolución": 587,
    "Bellas Artes": 447,
    "Guerrero": 702,
    "Juárez": 251
    },

    "Bellas Artes": {
        "Hidalgo": 447,
        "Allende": 387,
        "Garibaldi/Lagunilla": 634,
        "San Juan de Letrán": 456
    },

    "Allende": {
        "Bellas Artes": 387,
        "Zócalo/Tenochtitlan": 602
    },

    "Zócalo/Tenochtitlan": {
        "Allende": 602,
        "Pino Suárez": 745
    },

    "San Antonio Abad": {
        "Pino Suárez": 817,
        "Chabacano": 642
    },

    "Chabacano": {
        "San Antonio Abad": 642,
        "Viaducto": 774,
        "Obrera": 1143,
        "La Viga": 843,
        "Lázaro Cárdenas": 1000,
        "Jamaica": 1031
    },

    "Viaducto": {
        "Chabacano": 774,
        "Xola": 490
    },

    "Xola": {
        "Viaducto": 490,
        "Villa de Cortés": 698
    },

    "Villa de Cortés": {
        "Xola": 698,
        "Nativitas": 750
    },

    "Nativitas": {
        "Villa de Cortés": 750,
        "Portales": 924
    },

    "Portales": {
        "Nativitas": 924,
        "Ermita": 748
    },

    "Ermita": {
        "Portales": 748,
        "General Anaya": 838,
        "Mexicaltzingo": 1805,
        "Eje Central": 895
    },

    "General Anaya": {
        "Ermita": 838,
        "Tasqueña": 1330
    },

    "Tasqueña": {
        "General Anaya": 1330
    },

    # Línea 3

    "Indios Verdes": {
        "Deportivo 18 de Marzo": 1166
    },

    "Deportivo 18 de Marzo": {
        "Indios Verdes": 1166,
        "Potrero": 966,
        "Lindavista": 1075,
        "La Villa - Basílica": 570,
    },

    "Potrero": {
        "Deportivo 18 de Marzo": 966,
        "La Raza": 1106
    },

    "La Raza": {
        "Potrero": 1106,
        "Tlatelolco": 1445,
        "Autobuses del Norte": 975,
        "Misterios": 892
    },

    "Tlatelolco": {
        "La Raza": 1445,
        "Guerrero": 1042
    },

    "Guerrero": {
        "Tlatelolco": 1042,
        "Hidalgo": 702,
        "Garibaldi/Lagunilla": 757,
        "Buenavista": 521
    },

    "Juárez": {
        "Hidalgo": 251,
        "Balderas": 659
    },

    "Niños Héroes": {
        "Balderas": 665,
        "Hospital General": 559
    },

    "Hospital General": {
        "Niños Héroes": 559,
        "Centro Médico": 653
    },

    "Centro Médico": {
        "Hospital General": 653,
        "Etiopía/Plaza de la Transparencia": 1119,
        "Chilpancingo": 1152,
        "Lázaro Cárdenas": 1059
    },

    "Etiopía/Plaza de la Transparencia": {
        "Centro Médico": 1119,
        "Eugenia": 950
    },

    "Eugenia": {
        "Etiopía/Plaza de la Transparencia": 950,
        "División del Norte": 715
    },

    "División del Norte": {
        "Eugenia": 715,
        "Zapata": 794
    },

    "Zapata": {
        "División del Norte": 794,
        "Coyoacán": 1153,
        "Parque de los Venados": 563,
        "Hospital 20 de Noviembre": 450
    },

    "Coyoacán": {
        "Zapata": 1153,
        "Viveros/Derechos Humanos": 908
    },

    "Viveros/Derechos Humanos": {
        "Coyoacán": 908,
        "Miguel Ángel de Quevedo": 824
    },

    "Miguel Ángel de Quevedo": {
        "Viveros/Derechos Humanos": 824,
        "Copilco": 1295
    },

    "Copilco": {
        "Miguel Ángel de Quevedo": 1295,
        "Universidad": 1306
    },

    "Universidad": {
        "Copilco": 1306
    },

    # Línea 4

    "Santa Anita": {
        "Jamaica": 758,
        "La Viga": 633,
        "Coyuya": 968
    },

    "Jamaica": {
        "Santa Anita": 758,
        "Fray Servando": 1033,
        "Chabacano": 1031,
        "Mixiuhca": 942
    },

    "Fray Servando": {
        "Jamaica": 1033,
        "Candelaria": 633
    },

    "Morelos": {
        "Candelaria": 1062,
        "Canal del Norte": 910,
        "San Lázaro": 1296,
        "Tepito": 498
    },

    "Canal del Norte": {
        "Morelos": 910,
        "Consulado": 884
    },

    "Consulado": {
        "Canal del Norte": 884,
        "Bondojito": 645,
        "Valle Gómez": 679,
        "Eduardo Molina": 815
    },

    "Bondojito": {
        "Consulado": 645,
        "Talismán": 959
    },

    "Talismán": {
        "Bondojito": 959,
        "Martín Carrera": 1129
    },

    "Martín Carrera": {
        "Talismán": 1129,
        "La Villa - Basílica": 1141
    },

    # Línea 5

    "Politécnico": {
        "Instituto del Petróleo": 1188
    },

    "Instituto del Petróleo": {
        "Politécnico": 1188,
        "Autobuses del Norte": 1067,
        "Vallejo": 755,
        "Lindavista": 1258
    },

    "Autobuses del Norte": {
        "Instituto del Petróleo": 1067,
        "La Raza": 975
    },

    "Misterios": {
        "La Raza": 892,
        "Valle Gómez": 969
    },

    "Valle Gómez": {
        "Misterios": 969,
        "Consulado": 679
    },

    "Eduardo Molina": {
        "Consulado": 815,
        "Aragón": 860
    },

    "Aragón": {
        "Eduardo Molina": 860,
        "Oceanía": 1219
    },

    "Oceanía": {
        "Aragón": 1219,
        "Terminal Aérea": 1174,
        "Deportivo Oceanía": 863,
        "Romero Rubio": 809
    },

    "Terminal Aérea": {
        "Oceanía": 1174,
        "Hangares": 1153
    },

    "Hangares": {
        "Terminal Aérea": 1153,
        "Pantitlán": 1644
    },

    # Línea 6

    "El Rosario": {
        "Tezozómoc": 1257,
        "Aquiles Serdán": 1615
    },

    "Tezozómoc": {
        "El Rosario": 1257,
        "Azcapotzalco": 973
    },

    "Azcapotzalco": {
        "Tezozómoc": 973,
        "Ferrería": 1173
    },

    "Ferrería": {
        "Azcapotzalco": 1173,
        "Norte 45": 1072
    },

    "Norte 45": {
        "Ferrería": 1072,
        "Vallejo": 660
    },

    "Vallejo": {
        "Norte 45": 660,
        "Instituto del Petróleo": 755
    },

    "Lindavista": {
        "Instituto del Petróleo": 1258,
        "Deportivo 18 de Marzo": 1075
    },

    "La Villa - Basílica": {
        "Deportivo 18 de Marzo": 570,
        "Martín Carrera": 1141
    },

    # Línea 7

    "Aquiles Serdán": {
        "El Rosario": 1615,
        "Camarones": 1402
    },

    "Camarones": {
        "Aquiles Serdán": 1402,
        "Refinería": 952
    },

    "Refinería": {
        "Camarones": 952,
        "Tacuba": 1295
    },

    "San Joaquín": {
        "Tacuba": 1433,
        "Polanco": 1163
    },

    "Polanco": {
        "San Joaquín": 1163,
        "Auditorio": 812
    },

    "Auditorio": {
        "Polanco": 812,
        "Constituyentes": 1430
    },

    "Constituyentes": {
        "Auditorio": 1430,
        "Tacubaya": 1005
    },

    "San Pedro de los Pinos": {
        "Tacubaya": 1084,
        "San Antonio": 606
    },

    "San Antonio": {
        "San Pedro de los Pinos": 606,
        "Mixcoac": 788
    },

    "Mixcoac": {
        "San Antonio": 788,
        "Barranca del Muerto": 1476,
        "Insurgentes Sur": 651
    },

    "Barranca del Muerto": {
        "Mixcoac": 1476
    },

    # Línea 8

    "Garibaldi/Lagunilla": {
        "Bellas Artes": 634,
        "Lagunilla": 474,
        "Guerrero": 757
    },

    "San Juan de Letrán": {
        "Bellas Artes": 456,
        "Salto del Agua": 292
    },

    "Doctores": {
        "Salto del Agua": 564,
        "Obrera": 761
    },

    "Obrera": {
        "Doctores": 761,
        "Chabacano": 1143
    },

    "La Viga": {
        "Chabacano": 843,
        "Santa Anita": 633
    },

    "Coyuya": {
        "Santa Anita": 968,
        "Iztacalco": 993
    },

    "Iztacalco": {
        "Coyuya": 993,
        "Apatlaco": 910
    },

    "Apatlaco": {
        "Iztacalco": 910,
        "Aculco": 534
    },

    "Aculco": {
        "Apatlaco": 534,
        "Escuadrón 201": 789
    },

    "Escuadrón 201": {
        "Aculco": 789,
        "Atlalilco": 1738
    },

    "Atlalilco": {
        "Escuadrón 201": 1738,
        "Iztapalapa": 732,
        "Culhuacán": 1671,
        "Mexicaltzingo": 1922
    },

    "Iztapalapa": {
        "Atlalilco": 732,
        "Cerro de la Estrella": 717
    },

    "Cerro de la Estrella": {
        "Iztapalapa": 717,
        "UAM-I": 1135
    },

    "UAM-I": {
        "Cerro de la Estrella": 1135,
        "Constitución de 1917": 1137
    },

    "Constitución de 1917": {
        "UAM-I": 1137
    },

    # Línea 9

    "Patriotismo": {
        "Tacubaya": 1133,
        "Chilpancingo": 955
    },

    "Chilpancingo": {
        "Patriotismo": 955,
        "Centro Médico": 1152
    },

    "Lázaro Cárdenas": {
        "Centro Médico": 1059,
        "Chabacano": 1000
    },

    "Mixiuhca": {
        "Jamaica": 942,
        "Velódromo": 821
    },

    "Velódromo": {
        "Mixiuhca": 821,
        "Ciudad Deportiva": 1110
    },

    "Ciudad Deportiva": {
        "Velódromo": 1110,
        "Puebla": 800
    },

    "Puebla": {
        "Ciudad Deportiva": 800,
        "Pantitlán": 1380
    },

    # Línea A

    "Agrícola Oriental": {
        "Pantitlán": 1409,
        "Canal de San Juan": 1093
    },

    "Canal de San Juan": {
        "Agrícola Oriental": 1093,
        "Tepalcates": 1456
    },

    "Tepalcates": {
        "Canal de San Juan": 1456,
        "Guelatao": 1161
    },

    "Guelatao": {
        "Tepalcates": 1161,
        "Peñón Viejo": 2206
    },

    "Peñón Viejo": {
        "Guelatao": 2206,
        "Acatitla": 1379
    },

    "Acatitla": {
        "Peñón Viejo": 1379,
        "Santa Marta": 1100
    },

    "Santa Marta": {
        "Acatitla": 1100,
        "Los Reyes": 1783
    },

    "Los Reyes": {
        "Santa Marta": 1783,
        "La Paz": 1956
    },

    "La Paz": {
        "Los Reyes": 1956
    },

    # Línea B

    "Ciudad Azteca": {
        "Plaza Aragón": 574
    },

    "Plaza Aragón": {
        "Ciudad Azteca": 574,
        "Olímpica": 709
    },

    "Olímpica": {
        "Plaza Aragón": 709,
        "Ecatepec": 596
    },

    "Ecatepec": {
        "Olímpica": 596,
        "Múzquiz": 1485
    },

    "Múzquiz": {
        "Ecatepec": 1485,
        "Río de los Remedios": 1155
    },

    "Río de los Remedios": {
        "Múzquiz": 1155,
        "Impulsora": 436
    },

    "Impulsora": {
        "Río de los Remedios": 436,
        "Nezahualcóyotl": 1393
    },

    "Nezahualcóyotl": {
        "Impulsora": 1393,
        "Villa de Aragón": 1335
    },

    "Villa de Aragón": {
        "Nezahualcóyotl": 1335,
        "Bosques de Aragón": 784
    },

    "Bosques de Aragón": {
        "Villa de Aragón": 784,
        "Deportivo Oceanía": 1165
    },

    "Deportivo Oceanía": {
        "Bosques de Aragón": 1165,
        "Oceanía": 863
    },

    "Romero Rubio": {
        "Oceanía": 809,
        "Ricardo Flores Magón": 908
    },

    "Ricardo Flores Magón": {
        "Romero Rubio": 908,
        "San Lázaro": 907
    },

    "Tepito": {
        "Morelos": 498,
        "Lagunilla": 611
    },

    "Lagunilla": {
        "Tepito": 611,
        "Garibaldi/Lagunilla": 474
    },

    "Buenavista": {
        "Guerrero": 521
    },

    # Línea 12

    "Tláhuac": {
        "Tlaltenco": 1298
    },

    "Tlaltenco": {
        "Tláhuac": 1298,
        "Zapotitlán": 1115
    },

    "Zapotitlán": {
        "Tlaltenco": 1115,
        "Nopalera": 1276
    },

    "Nopalera": {
        "Zapotitlán": 1276,
        "Olivos": 1360
    },

    "Olivos": {
        "Nopalera": 1360,
        "Tezonco": 490
    },

    "Tezonco": {
        "Olivos": 490,
        "Periférico Oriente": 1545
    },

    "Periférico Oriente": {
        "Tezonco": 1545,
        "Calle 11": 1111
    },

    "Calle 11": {
        "Periférico Oriente": 1111,
        "Lomas Estrella": 906
    },

    "Lomas Estrella": {
        "Calle 11": 906,
        "San Andrés Tomatlán": 1060
    },

    "San Andrés Tomatlán": {
        "Lomas Estrella": 1060,
        "Culhuacán": 990
    },

    "Culhuacán": {
        "San Andrés Tomatlán": 990,
        "Atlalilco": 1671
    },

    "Mexicaltzingo": {
        "Atlalilco": 1922,
        "Ermita": 1805
    },

    "Eje Central": {
        "Ermita": 895,
        "Parque de los Venados": 1280
    },

    "Parque de los Venados": {
        "Eje Central": 1280,
        "Zapata": 563
    },

    "Hospital 20 de Noviembre": {
        "Zapata": 450,
        "Insurgentes Sur": 725
    },

    "Insurgentes Sur": {
        "Hospital 20 de Noviembre": 725,
        "Mixcoac": 651
    }

}

In [264]:
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def path(self):
        lista_path = []
        node = self

        while node:
            lista_path.append(node.state)
            node = node.parent

        return lista_path[::-1]

    def expand(self, problem):
        lista = []

        for action in problem.actions(self.state):
            lista.append(self.child_node(problem, action))

        return lista

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)

        step_cost = problem.action_cost(
            self.state,
            action,
            next_state
        )

        return Node(
            next_state,
            self,
            action,
            self.path_cost + step_cost
        )

In [265]:
# PARA COMPROBAR HARE UN CONTEO Y COMPROBACION DE UNA ESTACION
print(len(metro_cdmx))
print(metro_cdmx["Pantitlán"])

163
{'Zaragoza': 1320, 'Hangares': 1644, 'Puebla': 1380, 'Agrícola Oriental': 1409}


In [266]:
def depth_first_graph_search(problem):
    start_node = Node(problem.initial)

    if problem.is_goal(start_node.state):
        return start_node

    frontier = [start_node]
    explored = set()

    while frontier:
        node = frontier.pop()
        explored.add(node.state)

        if problem.is_goal(node.state):
            return node

        for child in node.expand(problem):
            if child.state not in explored:
                frontier.append(child)

    return None

In [267]:
from collections import deque

def breadth_first_graph_search(problem):
    start_node = Node(problem.initial)

    if problem.is_goal(start_node.state):
        return start_node

    frontier = deque([start_node])
    explored = set()

    while frontier:
        node = frontier.popleft()
        explored.add(node.state)

        if problem.is_goal(node.state):
            return node

        for child in node.expand(problem):
            if child.state not in explored:
                frontier.append(child)

    return None

In [268]:
problem = GraphProblem("Cuatro Caminos", "Pantitlán", metro_cdmx)

node_dfs = depth_first_graph_search(problem)

node_dfs.path()



['Cuatro Caminos',
 'Panteones',
 'Tacuba',
 'San Joaquín',
 'Polanco',
 'Auditorio',
 'Constituyentes',
 'Tacubaya',
 'Patriotismo',
 'Chilpancingo',
 'Centro Médico',
 'Lázaro Cárdenas',
 'Chabacano',
 'Jamaica',
 'Mixiuhca',
 'Velódromo',
 'Ciudad Deportiva',
 'Puebla',
 'Pantitlán']

In [269]:
node_dfs.path_cost

20281

In [270]:
problem = GraphProblem("Cuatro Caminos", "Pantitlán", metro_cdmx)

node_bfs = breadth_first_graph_search(problem)

node_bfs.path()

['Cuatro Caminos',
 'Panteones',
 'Tacuba',
 'San Joaquín',
 'Polanco',
 'Auditorio',
 'Constituyentes',
 'Tacubaya',
 'Patriotismo',
 'Chilpancingo',
 'Centro Médico',
 'Lázaro Cárdenas',
 'Chabacano',
 'Jamaica',
 'Mixiuhca',
 'Velódromo',
 'Ciudad Deportiva',
 'Puebla',
 'Pantitlán']

In [271]:
node_bfs.path_cost

20281

In [272]:
#USARE UNA FUNCION PARA MOSTRAR EL RESULTADO DE LA BUSQUEDA
def mostrar_resultado(nombre_metodo, nodo):
    ruta = nodo.path()
    distancia_km = nodo.path_cost / 1000

    print(f"=== {nombre_metodo} ===")
    print("Ruta encontrada:")
    print(" -> ".join(ruta))

    print(f"\nEstaciones recorridas: {len(ruta)}")
    print(f"Tramos recorridos: {len(ruta) - 1}")
    print(f"Con un total de {distancia_km:.3f} km")

# A PARTIR DE AQUÍ DE MUESTRAN LOS EJERCICIOS A PROBAR.
### CUATRO CAMINOS -> PANTITLÁN

In [273]:
problem = GraphProblem("Cuatro Caminos", "Pantitlán", metro_cdmx)

node_dfs = depth_first_graph_search(problem)

mostrar_resultado("DFS", node_dfs)

=== DFS ===
Ruta encontrada:
Cuatro Caminos -> Panteones -> Tacuba -> San Joaquín -> Polanco -> Auditorio -> Constituyentes -> Tacubaya -> Patriotismo -> Chilpancingo -> Centro Médico -> Lázaro Cárdenas -> Chabacano -> Jamaica -> Mixiuhca -> Velódromo -> Ciudad Deportiva -> Puebla -> Pantitlán

Estaciones recorridas: 19
Tramos recorridos: 18
Con un total de 20.281 km


In [274]:
problem = GraphProblem("Cuatro Caminos", "Pantitlán", metro_cdmx)

node_bfs = breadth_first_graph_search(problem)

mostrar_resultado("BFS", node_bfs)

=== BFS ===
Ruta encontrada:
Cuatro Caminos -> Panteones -> Tacuba -> San Joaquín -> Polanco -> Auditorio -> Constituyentes -> Tacubaya -> Patriotismo -> Chilpancingo -> Centro Médico -> Lázaro Cárdenas -> Chabacano -> Jamaica -> Mixiuhca -> Velódromo -> Ciudad Deportiva -> Puebla -> Pantitlán

Estaciones recorridas: 19
Tramos recorridos: 18
Con un total de 20.281 km


### POLITECNICO -> TASQUEÑA

In [275]:
problem = GraphProblem("Politécnico", "Tasqueña", metro_cdmx)

node_dfs = depth_first_graph_search(problem)

mostrar_resultado("DFS", node_dfs)

=== DFS ===
Ruta encontrada:
Politécnico -> Instituto del Petróleo -> Lindavista -> Deportivo 18 de Marzo -> La Villa - Basílica -> Martín Carrera -> Talismán -> Bondojito -> Consulado -> Eduardo Molina -> Aragón -> Oceanía -> Romero Rubio -> Ricardo Flores Magón -> San Lázaro -> Morelos -> Tepito -> Lagunilla -> Garibaldi/Lagunilla -> Guerrero -> Hidalgo -> Juárez -> Balderas -> Niños Héroes -> Hospital General -> Centro Médico -> Lázaro Cárdenas -> Chabacano -> Jamaica -> Santa Anita -> Coyuya -> Iztacalco -> Apatlaco -> Aculco -> Escuadrón 201 -> Atlalilco -> Mexicaltzingo -> Ermita -> General Anaya -> Tasqueña

Estaciones recorridas: 40
Tramos recorridos: 39
Con un total de 36.283 km


In [276]:
problem = GraphProblem("Politécnico", "Tasqueña", metro_cdmx)

node_bfs = breadth_first_graph_search(problem)

mostrar_resultado("BFS", node_bfs)

=== BFS ===
Ruta encontrada:
Politécnico -> Instituto del Petróleo -> Autobuses del Norte -> La Raza -> Tlatelolco -> Guerrero -> Hidalgo -> Bellas Artes -> Allende -> Zócalo/Tenochtitlan -> Pino Suárez -> San Antonio Abad -> Chabacano -> Viaducto -> Xola -> Villa de Cortés -> Nativitas -> Portales -> Ermita -> General Anaya -> Tasqueña

Estaciones recorridas: 21
Tramos recorridos: 20
Con un total de 16.611 km


### Zapata -> Oceanía

In [277]:
problem = GraphProblem("Zapata", "Oceanía", metro_cdmx)

node_dfs = depth_first_graph_search(problem)

mostrar_resultado("DFS", node_dfs)

=== DFS ===
Ruta encontrada:
Zapata -> Hospital 20 de Noviembre -> Insurgentes Sur -> Mixcoac -> San Antonio -> San Pedro de los Pinos -> Tacubaya -> Patriotismo -> Chilpancingo -> Centro Médico -> Lázaro Cárdenas -> Chabacano -> Jamaica -> Mixiuhca -> Velódromo -> Ciudad Deportiva -> Puebla -> Pantitlán -> Hangares -> Terminal Aérea -> Oceanía

Estaciones recorridas: 21
Tramos recorridos: 20
Con un total de 19.658 km


In [278]:
problem = GraphProblem("Zapata", "Oceanía", metro_cdmx)

node_bfs = breadth_first_graph_search(problem)

mostrar_resultado("BFS", node_bfs)

=== BFS ===
Ruta encontrada:
Zapata -> División del Norte -> Eugenia -> Etiopía/Plaza de la Transparencia -> Centro Médico -> Lázaro Cárdenas -> Chabacano -> Jamaica -> Fray Servando -> Candelaria -> San Lázaro -> Ricardo Flores Magón -> Romero Rubio -> Oceanía

Estaciones recorridas: 14
Tramos recorridos: 13
Con un total de 11.824 km


### Hacemos una tabla comparativa

In [279]:
import pandas as pd
resultados = {
    "Recorrido": [
        "Cuatro Caminos → Pantitlán",
        "Cuatro Caminos → Pantitlán",
        "Politécnico → Tasqueña",
        "Politécnico → Tasqueña",
        "Zapata → Oceanía",
        "Zapata → Oceanía"
    ],

    "Método": [
        "DFS",
        "BFS",
        "DFS",
        "BFS",
        "DFS",
        "BFS"
    ],

    "Estaciones": [
        19,
        19,
        40,
        21,
        21,
        14
    ],

    "Tramos": [
        18,
        18,
        39,
        20,
        20,
        13
    ],

    "Distancia (km)": [
        20.281,
        20.281,
        36.283,
        16.611,
        19.658,
        11.824
    ]
}

tabla_resultados = pd.DataFrame(resultados)

tabla_resultados

,Recorrido,Método,Estaciones,Tramos,Distancia (km)
0,Cuatro Caminos → Pantitlán,DFS,19,18,20.281
1,Cuatro Caminos → Pantitlán,BFS,19,18,20.281
2,Politécnico → Tasqueña,DFS,40,39,36.283
3,Politécnico → Tasqueña,BFS,21,20,16.611
4,Zapata → Oceanía,DFS,21,20,19.658
5,Zapata → Oceanía,BFS,14,13,11.824


## Conclusión

En este ejercicio, BFS resultó ser más conveniente que DFS, ya que encontró rutas más cortas en la mayoría de los casos. DFS también logró llegar al destino, pero en algunos recorridos tomó caminos mucho más largos.

Aunque BFS no usa directamente la distancia en kilómetros para decidir la ruta, en estos ejemplos dio mejores resultados que DFS.